In [35]:
# @title Secure API Initialization
# ============================================
# IMPORTS
# ============================================

import gradio as gr
from google import genai  # Modern upgraded SDK import
import json
import re
import os
from datetime import datetime, timedelta
from typing import Dict, List, Any
from google.colab import userdata

print("✅ All imports loaded!")

# ============================================
# SET GEMINI API KEY VIA COLAB SECRETS ONLY
# ============================================

GEMINI_API_KEY = None

try:
    # Pull securely from your Colab Secrets vault using your custom name 'SamAPI'
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception as e:
    print(f"⚠️ Could not access Colab Secrets: {e}")

# ============================================
# INITIALIZE GLOBAL CLIENT OBJECT
# ============================================

if GEMINI_API_KEY:
    # This creates the exact 'ai_client' object your orchestrator and agents are searching for!
    ai_client = genai.Client(api_key=GEMINI_API_KEY)
    print("✅ Global object 'ai_client' successfully created securely via Secrets!")
else:
    print("\n❌ CRITICAL ERROR: GEMINI_API_KEY not found.")
    print("👉 Fix: Click the 🔑 (Secrets) tab on the left sidebar in Colab.")
    print("👉 Make sure there is a secret named EXACTLY: SamAPI")
    print("👉 Turn the 'Notebook access' toggle switch to ON.")
    exit(1)

✅ All imports loaded!
✅ Global object 'ai_client' successfully created securely via Secrets!


In [36]:
# @title
# ============================================
# LOAD PRODUCT_CATALOG DIRECTLY FROM LIVE GOOGLE SHEET
# ============================================

import pandas as pd
# Paste your modified export link here
SHEET_URL = "https://docs.google.com/spreadsheets/d/1iwZm2QOijFqPKsqnK5fXutGzflLvPB-ujb_wgHnsBjo/export?format=csv"


# Read directly from the live cloud sheet! No drive mounting required.
df = pd.read_csv(SHEET_URL)

# Convert to Dictionary (The rest of your code stays exactly the same)
PRODUCT_CATALOG = {}

for category in df['category'].unique():
    category_df = df[df['category'] == category]
    products = []
    for _, row in category_df.iterrows():
        product = {
            'name': row['name'],
            'type': row['type'],
            'price': int(row['price']) if pd.notna(row['price']) else 0,
        }

        # Optional fields check
        if pd.notna(row.get('socket')): product['socket'] = row['socket']
        if pd.notna(row.get('cores')): product['cores'] = int(row['cores'])
        if pd.notna(row.get('threads')): product['threads'] = int(row['threads'])
        if pd.notna(row.get('tdp')): product['tdp'] = int(row['tdp'])
        if pd.notna(row.get('vram')): product['vram'] = int(row['vram'])
        if pd.notna(row.get('ram_type')): product['ram_type'] = row['ram_type']
        if pd.notna(row.get('wattage')): product['wattage'] = int(row['wattage'])
        if pd.notna(row.get('stock')):
            product['stock'] = int(row['stock'])
        else:
            product['stock'] = 15

        products.append(product)
    PRODUCT_CATALOG[category] = products

print("="*50)
print("✅ PRODUCT_CATALOG loaded live from Google Sheets!")
print(f"   Categories synced: {list(PRODUCT_CATALOG.keys())}")
print("="*50)

✅ PRODUCT_CATALOG loaded live from Google Sheets!
   Categories synced: ['CPUs', 'GPUs', 'Motherboards', 'RAM', 'Storage', 'PSU', 'Cases', 'Coolers']


In [37]:
# @title
# ============================================
# LOAD PRODUCT_CATALOG FROM CSV AND CONVERT TO DICTIONARY
# ============================================

import pandas as pd
str: SHEET_URL


# Read the CSV file
df = pd.read_csv(SHEET_URL)

# Convert to Dictionary (what your code expects)
PRODUCT_CATALOG = {}

for category in df['category'].unique():
    # Get all rows for this category
    category_df = df[df['category'] == category]

    # Convert to list of dictionaries
    products = []
    for _, row in category_df.iterrows():
        product = {
            'name': row['name'],
            'type': row['type'],
            'price': int(row['price']) if pd.notna(row['price']) else 0,
        }

        # Add optional fields (only if they exist)
        if pd.notna(row.get('socket')):
            product['socket'] = row['socket']
        if pd.notna(row.get('cores')):
            product['cores'] = int(row['cores'])
        if pd.notna(row.get('threads')):
            product['threads'] = int(row['threads'])
        if pd.notna(row.get('tdp')):
            product['tdp'] = int(row['tdp'])
        if pd.notna(row.get('gaming_performance')):
            product['gaming_performance'] = int(row['gaming_performance'])
        if pd.notna(row.get('vram')):
            product['vram'] = int(row['vram'])
        if pd.notna(row.get('ram_type')):
            product['ram_type'] = row['ram_type']
        if pd.notna(row.get('wattage')):
            product['wattage'] = int(row['wattage'])
        if pd.notna(row.get('capacity')):
            product['capacity'] = int(row['capacity']) if isinstance(row['capacity'], (int, float)) else row['capacity']
        if pd.notna(row.get('speed')):
            product['speed'] = row['speed']
        if pd.notna(row.get('features')):
            product['features'] = row['features'].split(', ') if isinstance(row['features'], str) else [row['features']]
        if pd.notna(row.get('fans_included')):
            product['fans_included'] = int(row['fans_included'])
        if pd.notna(row.get('tdp_rating')):
            product['tdp_rating'] = int(row['tdp_rating'])
 # ========== ADD STOCK SUPPORT ==========
        if pd.notna(row.get('stock')):
            product['stock'] = int(row['stock'])
        else:
            product['stock'] = 15  # Default stock if column missing

        products.append(product)

    PRODUCT_CATALOG[category] = products

# Verify it worked
print("="*50)
print("✅ PRODUCT_CATALOG loaded from CSV!")
print(f"   Type: {type(PRODUCT_CATALOG)}")
print(f"   Categories: {list(PRODUCT_CATALOG.keys())}")
print(f"   CPUs: {len(PRODUCT_CATALOG.get('CPUs', []))} items")
print(f"   GPUs: {len(PRODUCT_CATALOG.get('GPUs', []))} items")
print(f"   Motherboards: {len(PRODUCT_CATALOG.get('Motherboards', []))} items")
print(f"   RAM: {len(PRODUCT_CATALOG.get('RAM', []))} items")
print(f"   Storage: {len(PRODUCT_CATALOG.get('Storage', []))} items")
print(f"   PSU: {len(PRODUCT_CATALOG.get('PSU', []))} items")
print(f"   Cases: {len(PRODUCT_CATALOG.get('Cases', []))} items")
print(f"   Coolers: {len(PRODUCT_CATALOG.get('Coolers', []))} items")
for category in ['CPUs', 'GPUs', 'RAM']:
    if PRODUCT_CATALOG.get(category):
        for product in PRODUCT_CATALOG[category][:2]:
            print(f"   {product['name']}: Stock = {product.get('stock', 'MISSING')}")
print("="*50)

✅ PRODUCT_CATALOG loaded from CSV!
   Type: <class 'dict'>
   Categories: ['CPUs', 'GPUs', 'Motherboards', 'RAM', 'Storage', 'PSU', 'Cases', 'Coolers']
   CPUs: 4 items
   GPUs: 4 items
   Motherboards: 4 items
   RAM: 4 items
   Storage: 4 items
   PSU: 4 items
   Cases: 3 items
   Coolers: 3 items
   Ryzen 5 5600X: Stock = 10
   Ryzen 5 7600: Stock = 30
   GTX 1650: Stock = 23
   RTX 3060: Stock = 18
   16GB DDR4 RAM: Stock = 25
   32GB DDR4 RAM: Stock = 16


In [38]:
# @title
# ============================================
# LOAD DELIVERY LOCATIONS
# ============================================

import pandas as pd


# 1. Paste your modified delivery sheet export link here
DELIVERY_SHEET_URL = "https://docs.google.com/spreadsheets/d/14ni7B6HeecsIN77hBnUgRksQwO86mHWD5CO6vJ5cLyw/export?format=csv"

# Just read normally - pandas auto-detects commas
df_delivery = pd.read_csv(DELIVERY_SHEET_URL)

# Clean column names (remove any spaces)
df_delivery.columns = df_delivery.columns.str.strip()

DELIVERY_LOCATIONS = {}

for _, row in df_delivery.iterrows():
    DELIVERY_LOCATIONS[row['location']] = {
        'available': str(row['available']).upper() == 'TRUE',
        'base_days': int(row['base_days']),
        'shipping_cost': int(row['shipping_cost']),
        'zones': [z.strip() for z in row['zones'].split(',')]
    }

print(f"✅ Loaded {len(DELIVERY_LOCATIONS)} delivery locations")
print(f"   Columns: {list(df_delivery.columns)}")
print(f"   First location: {list(DELIVERY_LOCATIONS.keys())[0]}")



# Step 7: Test accessing data (exactly how your code will use it)
print("\n6. Testing access (simulating your code):")
test_location = 'Kuala Lumpur'

if test_location in DELIVERY_LOCATIONS:
    info = DELIVERY_LOCATIONS[test_location]
    print(f"   ✅ '{test_location}' found!")
    print(f"   📍 base_days: {info['base_days']}")
    print(f"   💰 shipping_cost: RM{info['shipping_cost']}")
    print(f"   🗺️  zones: {info['zones'][:3]}")
    print(f"   ✅ available: {info['available']}")
else:
    print(f"   ❌ '{test_location}' not found!")
    print(f"   Available locations: {list(DELIVERY_LOCATIONS.keys())[:5]}")

# Step 8: Test multiple locations
print("\n7. Testing multiple locations:")
test_locations = ['George Town', 'Johor Bahru', 'Ipoh', 'Kuching']
for loc in test_locations:
    if loc in DELIVERY_LOCATIONS:
        info = DELIVERY_LOCATIONS[loc]
        print(f"   ✅ {loc}: {info['base_days']} days, RM{info['shipping_cost']}")
    else:
        print(f"   ❌ {loc} not found")

# Step 9: Final verification
print("\n" + "="*60)
print("TEST RESULTS")
print("="*60)

# Check if your QuotationAgent would work
try:
    sample = DELIVERY_LOCATIONS['Kuala Lumpur']
    test_zones = sample['zones']
    test_days = sample['base_days']
    test_cost = sample['shipping_cost']
    print("✅ DELIVERY_LOCATIONS is READY for your QuotationAgent!")
    print(f"\n   Example output would be:")
    print(f"   Delivery to Kuala Lumpur: {test_days} days")
    print(f"   Shipping cost: RM{test_cost}")
    print(f"   Zones: {', '.join(test_zones[:3])}")
except Exception as e:
    print(f"❌ Error: {e}")

print("\n" + "="*60)


# ============================================
# LOCATION NORMALIZATION (Map user input to database locations)
# ============================================

LOCATION_MAPPING = {
    # Penang area
    'penang': 'George Town',
    'pg': 'George Town',
    'george town': 'George Town',
    'georgetown': 'George Town',
    'butterworth': 'Butterworth',
    'bukit mertajam': 'Bukit Mertajam',
    'bayan lepas': 'Bayan Lepas',

    # Klang Valley
    'kl': 'Kuala Lumpur',
    'kuala lumpur': 'Kuala Lumpur',
    'putrajaya': 'Putrajaya',
    'selangor': 'Shah Alam',
    'shah alam': 'Shah Alam',
    'petaling jaya': 'Petaling Jaya',
    'subang jaya': 'Subang Jaya',
    'puchong': 'Puchong',
    'cyberjaya': 'Cyberjaya',
    'klang': 'Klang',

    # Johor
    'jb': 'Johor Bahru',
    'johor bahru': 'Johor Bahru',
    'johor': 'Johor Bahru',
    'iskandar puteri': 'Iskandar Puteri',
    'batu pahat': 'Batu Pahat',
    'muar': 'Muar',
    'kluang': 'Kluang',

    # East Malaysia
    'kk': 'Kota Kinabalu',
    'kota kinabalu': 'Kota Kinabalu',
    'sabah': 'Kota Kinabalu',
    'sandakan': 'Sandakan',
    'tawau': 'Tawau',
    'kuching': 'Kuching',
    'sarawak': 'Kuching',
    'sibu': 'Sibu',
    'miri': 'Miri',
    'bintulu': 'Bintulu',

    # Other states
    'ipoh': 'Ipoh',
    'perak': 'Ipoh',
    'taiping': 'Taiping',
    'teluk intan': 'Teluk Intan',
    'alor setar': 'Alor Setar',
    'kedah': 'Alor Setar',
    'sungai petani': 'Sungai Petani',
    'langkawi': 'Langkawi',
    'melaka': 'Malacca City',
    'malacca': 'Malacca City',
    'kuantan': 'Kuantan',
    'pahang': 'Kuantan',
    'kota bharu': 'Kota Bharu',
    'kelantan': 'Kota Bharu',
    'kuala terengganu': 'Kuala Terengganu',
    'terengganu': 'Kuala Terengganu',
    'seremban': 'Seremban',
    'negeri sembilan': 'Seremban',
    'port dickson': 'Port Dickson',
    'kangar': 'Kangar',
    'perlis': 'Kangar',
}

def normalize_location(user_location):
    """Convert user's location input to actual location name in DELIVERY_LOCATIONS"""
    if not user_location:
        return None

    user_loc_lower = user_location.lower().strip()

    # Direct match
    for loc in DELIVERY_LOCATIONS.keys():
        if loc.lower() == user_loc_lower:
            return loc

    # Alias mapping
    if user_loc_lower in LOCATION_MAPPING:
        return LOCATION_MAPPING[user_loc_lower]

    # Partial match
    for loc in DELIVERY_LOCATIONS.keys():
        if user_loc_lower in loc.lower() or loc.lower() in user_loc_lower:
            return loc

    return None

✅ Loaded 37 delivery locations
   Columns: ['location', 'state', 'available', 'base_days', 'shipping_cost', 'zones']
   First location: Kuala Lumpur

6. Testing access (simulating your code):
   ✅ 'Kuala Lumpur' found!
   📍 base_days: 1
   💰 shipping_cost: RM20
   🗺️  zones: ['KLCC', 'Bukit Bintang', 'Cheras']
   ✅ available: True

7. Testing multiple locations:
   ✅ George Town: 2 days, RM30
   ✅ Johor Bahru: 3 days, RM35
   ✅ Ipoh: 2 days, RM30
   ✅ Kuching: 5 days, RM80

TEST RESULTS
✅ DELIVERY_LOCATIONS is READY for your QuotationAgent!

   Example output would be:
   Delivery to Kuala Lumpur: 1 days
   Shipping cost: RM20
   Zones: KLCC, Bukit Bintang, Cheras



In [39]:
# ============================================
# AGENT 1: REQUIREMENT AGENT (FIXED FALLBACK)
# ============================================

class RequirementAgent:
    """Extracts structured fields from raw user prompts using Gemini"""

    def extract_requirements(self, user_input: str) -> Dict[str, Any]:
        # Initialize default values safely to capture user text before entering try block
        extracted_budget = 4000
        detected_location = "Outside Delivery Zone"

        try:
            # 1. Stably parse out numerical budget figures
            budget_match = re.search(r'(?:RM|budget of|around)\s*(\d+[\d,.]*)', user_input, re.IGNORECASE)
            if budget_match:
                cleaned_b = budget_match.group(1).replace(',', '')
                extracted_budget = int(float(cleaned_b))

            # 2. Check if the input contains a covered location from our dictionary space
            global DELIVERY_LOCATIONS
            has_covered_location = False
            if 'DELIVERY_LOCATIONS' in globals() and DELIVERY_LOCATIONS:
                for loc in DELIVERY_LOCATIONS.keys():
                    if loc.lower() in user_input.lower():
                        detected_location = loc
                        has_covered_location = True
                        break

            # 3. Dynamic Regex Fallback: If it's not a covered location, look for common patterns
            # like "deliver to [Location]" or "delivered to [Location]" to capture unserviced regions (e.g. Korea)
            if not has_covered_location:
                location_match = re.search(r'(?:deliver to|delivered to|in|at)\s+([A-Z][a-zA-Z\s]{2,19})', user_input, re.IGNORECASE)
                if location_match:
                    detected_location = location_match.group(1).strip()
                else:
                    # Clean sentence cleanups to grab the final word if it looks like a location token
                    words = user_input.strip().rstrip('.!?').split()
                    if words:
                        detected_location = words[-1].title()

            prompt = f"""
            Analyze this PC buyer request: "{user_input}"

            Extract the values into a strict JSON dictionary with these exact keys:
            - "budget": integer value
            - "location": text string (match exact spelling of the user's intended country or city string: e.g. "Korea")
            - "use_case": text string ("gaming", "workstation", or "balanced")
            - "delivery_available": boolean (true ONLY if the location is one of these exact keys: {list(DELIVERY_LOCATIONS.keys()) if 'DELIVERY_LOCATIONS' in globals() else []}, false otherwise)

            Return ONLY the raw valid JSON code block. No extra text explanation.
            """

            response = ai_client.models.generate_content(
                model='gemini-3.5-flash',
                contents=prompt,
            )

            cleaned_text = response.text.replace("```json", "").replace("```", "").strip()
            data = json.loads(cleaned_text)
            return data

        except Exception as e:
            print(f"⚠️ Requirement parsing exception: {e}")
            # FIXED: Returns the true parsed location string instead of forcing Kuala Lumpur
            return {
                "budget": extracted_budget,
                "location": detected_location,
                "use_case": "gaming",
                "delivery_available": False
            }

In [40]:
# ============================================
# AGENT 2: SEARCH AGENT (FIXED BASELINE OVERWRITE)
# ============================================

class SearchAgent:
    """Retrieves available component pools matching user profiles from master inventories"""

    def search_components(self, requirements: Dict[str, Any]) -> List[Dict[str, Any]]:
        global PRODUCT_CATALOG
        selected_components = []

        if not PRODUCT_CATALOG:
            print("❌ Master product catalog is empty.")
            return selected_components

        # Extract one baseline operational part per category to feed the pipeline state stably
        for category, items in PRODUCT_CATALOG.items():
            in_stock_items = [p for p in items if p.get('stock', 0) > 0]
            if in_stock_items:
                # Sort by price ascending to provide a clean baseline minimum reference build
                sorted_items = sorted(in_stock_items, key=lambda x: x.get('price', 0))
                selected_components.append(sorted_items[0])

        # Safe normalization mapping names back to your Sheets index layout keys
        for comp in selected_components:
            if 'stock' not in comp: comp['stock'] = 15
            if comp.get('type') == 'CPU': comp['type'] = 'CPUs'
            if comp.get('type') == 'GPU': comp['type'] = 'GPUs'
            if comp.get('type') == 'Case': comp['type'] = 'Cases'
            if comp.get('type') == 'Cooler': comp['type'] = 'Coolers'
            if comp.get('type') == 'Motherboard': comp['type'] = 'Motherboards'

        return selected_components

In [41]:


# ============================================
# AGENT 3: COMPATIBILITY AGENT
# ============================================

class CompatibilityAgent:
    def check_compatibility(self, components: List[Dict]) -> Dict:
        issues = []
        warnings = []

        comp_by_type = {c["type"]: c for c in components}

        # CPU-Motherboard compatibility
        if "CPU" in comp_by_type and "Motherboard" in comp_by_type:
            cpu = comp_by_type["CPU"]
            mb = comp_by_type["Motherboard"]
            if cpu.get("socket") != mb.get("socket"):
                issues.append(f"CPU socket {cpu.get('socket')} incompatible with motherboard socket {mb.get('socket')}")

        # RAM-Motherboard compatibility
        if "RAM" in comp_by_type and "Motherboard" in comp_by_type:
            ram = comp_by_type["RAM"]
            mb = comp_by_type["Motherboard"]
            if "DDR5" in ram["ram_type"] and "DDR5" not in str(mb.get("features", [])):
                warnings.append("DDR5 RAM may not be compatible with motherboard")

        # PSU wattage
        if "PSU" in comp_by_type:
            psu = comp_by_type["PSU"]
            total_tdp = sum(c.get("tdp", 65) for c in components if "tdp" in c)
            if psu["wattage"] < total_tdp:
                issues.append(f"PSU {psu['wattage']}W insufficient for total TDP {total_tdp}W")

        return {
            "compatible": len(issues) == 0,
            "issues": issues,
            "warnings": warnings,
            "status": "Compatible Build" if len(issues) == 0 else "Incompatible Build"
        }

In [42]:
# =====================================================
# AGENT 4: OPTIMIZER AGENT
# =====================================================


from typing import List, Dict

class OptimizerAgent:
    def __init__(self, get_catalog_items_func=None):
        """
        Initializes the Optimizer Agent Core with a defensive default fallback.
        """
        # If a function is provided, use it. Otherwise, fall back to a default global lookup function or pass.
        if get_catalog_items_func is not None:
            self._get_catalog_items = get_catalog_items_func
        else:
            # Fallback default: binds to a global catalog function if it exists in your notebook
            try:
                self._get_catalog_items = get_catalog_items
            except NameError:
                # If your notebook uses a different global loader name, use it here
                self._get_catalog_items = lambda category: []

    def _calculate_system_performance(self, cpu: Dict, gpu: Dict, use_case: str = "general") -> float:
        """
        Calculates the Unified Performance Index (P_system) using dynamic,
        workload-specific weight vectors mapping to real-world computer engineering constraints.
        """
        # 1. Extract raw normalized performance benchmarks from current spreadsheet entries
        r_cpu = float(cpu.get("performance_score", cpu.get("cpu_performance", 50)))
        r_gpu = float(gpu.get("gaming_performance", gpu.get("gpu_performance", 50)))

        # 2. Normalize user intent payload string to eliminate casing mismatches
        intent = str(use_case).lower() if use_case else "general"

        # Profile A: Graphic-Intensive / Gaming Focus (GPU drives, CPU assists)
        if any(k in intent for k in ["gaming", "fps", "play", "aaa", "graphics"]):
            w_gpu, w_cpu = 0.6, 0.4

        # Profile B: Compute-Intensive / Workstation Focus (CPU drives, GPU assists)
        elif any(k in intent for k in ["excel", "word", "office", "presentation", "working", "programming"]):
            w_cpu, w_gpu = 0.5, 0.5

        # Profile C: General Purpose / Universal Balanced Fallback
        else:
            w_cpu, w_gpu = 0.5, 0.5

        # 3. Compute scalar system performance value (Vector Sum equals 1.0)
        p_system = (w_cpu * r_cpu) + (w_gpu * r_gpu)
        return round(p_system, 2)

    def _pick_best_by_performance(self, items: List[Dict], key_name: str) -> Dict:
        """Isolates the single highest-performing hardware variant via its metric string key."""
        if not items:
            return {}
        return sorted(items, key=lambda x: float(x.get(key_name, 0)), reverse=True)[0]

    def _pick_cheapest(self, items: List[Dict]) -> Dict:
        """Locates the absolute financial floor entry out of the filtered candidate matrix."""
        if not items:
            return {}
        return sorted(items, key=lambda x: float(x.get("price", 0)))[0]

    def _pick_best_value(self, items: List[Dict], metric_key: str) -> Dict:
        """Calculates the maximum tangent efficiency coordinate (Performance / Price)."""
        if not items:
            return {}
        return sorted(
            items,
            key=lambda x: float(x.get(metric_key, 0)) / max(float(x.get("price", 1)), 1),
            reverse=True
        )[0]

    def _find_compatible_motherboards(self, cpu: Dict) -> List[Dict]:
        """Runs deterministic socket aggregation heuristics to filter matching motherboards."""
        motherboards = self._get_catalog_items("Motherboards")
        cpu_socket = cpu.get("socket", "")
        return [mb for mb in motherboards if mb.get("socket", "") == cpu_socket] or motherboards

    def _find_compatible_ram(self, motherboard: Dict) -> List[Dict]:
        """Runs memory generation heuristics to block DDR4/DDR5 cross-generation installation faults."""
        rams = self._get_catalog_items("RAM")
        mb_features = str(motherboard.get("features", "")).upper()

        if "DDR5" in mb_features:
            return [r for r in rams if "DDR5" in str(r.get("ram_generation", r.get("name", "")))] or rams
        elif "DDR4" in mb_features:
            return [r for r in rams if "DDR4" in str(r.get("ram_generation", r.get("name", "")))] or rams
        return rams

    def _find_compatible_psu(self, cpu: Dict, gpu: Dict) -> List[Dict]:
        """
        Calculates dynamic required system wattage thresholds using
        our strict 25% protective engineering headroom cushion rule.
        """
        psus = self._get_catalog_items("PSU")

        # Establish dynamic lower boundary threshold gate: (TDP Sum) * 1.25
        required_wattage = int((float(cpu.get("tdp", 65)) + float(gpu.get("tdp", 75))) * 1.25)

        # Aggregation filter: exclude unsafe power supplies
        compatible = [p for p in psus if float(p.get("wattage", 0)) >= required_wattage]
        return compatible if compatible else psus

    def optimize_build(self, strategy: str, budget: int, use_case: str = "general") -> Dict:
        """
        Public execution alias matching the orchestrator interface call.
        Routes directly into the Double-Filtration pipeline handler with an optional default use_case.
        """
        return self._build_candidate(strategy=strategy, budget=budget, use_case=use_case)

    def _build_candidate(self, strategy: str, budget: int, use_case: str = "general") -> Dict:
        """
        The Main Double-Filtration Optimization Engine.
        Enforces structural budget allocation caps before executing inner Pareto matrix sorts.
        """
        cpus = self._get_catalog_items("CPUs")
        gpus = self._get_catalog_items("GPUs")
        storages = self._get_catalog_items("Storage")
        cases = self._get_catalog_items("Cases")
        coolers = self._get_catalog_items("Coolers")

        if not cpus or not gpus:
            return {}

        strategy_str = str(strategy).lower()

        # =====================================================================
        # DOUBLE FILTRATION BLOCK: PROCESSORS & GRAPHICS INFRASTRUCTURE
        # =====================================================================
        if strategy_str == "max_performance":
            # Filtration Pass 1: Apply strict 35% / 55% sub-budget resource allocations
            valid_cpus = [c for c in cpus if float(c.get("price", 0)) <= budget * 0.35] or cpus
            valid_gpus = [g for g in gpus if float(g.get("price", 0)) <= budget * 0.55] or gpus
            # Filtration Pass 2: Saturate maximum possible component benchmark curves
            cpu = self._pick_best_by_performance(valid_cpus, "performance_score")
            gpu = self._pick_best_by_performance(valid_gpus, "gaming_performance")

        elif strategy_str == "budget_saver":
            # Filtration Pass 1 & 2: Directly grab the curated database baseline price floor
            cpu = self._pick_cheapest(cpus)
            gpu = self._pick_cheapest(gpus)

        else:  # optimal_value Track
            # Filtration Pass 1: Restrict components to highly efficient 25% / 40% ceilings
            valid_cpus = [c for c in cpus if float(c.get("price", 0)) <= budget * 0.25] or cpus
            valid_gpus = [g for g in gpus if float(g.get("price", 0)) <= budget * 0.40] or gpus
            # Filtration Pass 2: Maximize Tangent Efficiency Ratios (Performance divided by Price)
            cpu = self._pick_best_value(valid_cpus, "performance_score")
            gpu = self._pick_best_value(valid_gpus, "gaming_performance")

        # =====================================================================
        # AGGREGATION HEURISTICS BLOCK: COMPATIBLE BACKBONE GENERATION
        # =====================================================================
        # 1. Match Motherboard Form-Factor (Socket Verification check)
        valid_mbs = self._find_compatible_motherboards(cpu)
        motherboard = self._pick_best_by_performance(valid_mbs, "performance_score") if strategy_str == "max_performance" else self._pick_cheapest(valid_mbs)

        # 2. Match RAM Generations (DDR4 / DDR5 structural flag matching)
        valid_rams = self._find_compatible_ram(motherboard)
        ram = sorted(valid_rams, key=lambda x: float(x.get("capacity", 0)), reverse=True)[0] if strategy_str == "max_performance" else self._pick_cheapest(valid_rams)

        # 3. Match Electrical Capacity (Dynamic 25% safety overhead filter)
        valid_psus = self._find_compatible_psu(cpu, gpu)
        psu = self._pick_best_by_performance(valid_psus, "wattage") if strategy_str == "max_performance" else self._pick_cheapest(valid_psus)

        # 4. Extract Remaining Secondary Equipment Assets
        storage = sorted(storages, key=lambda x: float(x.get("capacity", 0)), reverse=True)[0] if strategy_str == "max_performance" else self._pick_cheapest(storages)
        case = self._pick_best_by_performance(cases, "airflow_rating") if strategy_str == "max_performance" else self._pick_cheapest(cases)
        cooler = self._pick_best_by_performance(coolers, "cooling_efficiency") if strategy_str == "max_performance" else self._pick_cheapest(coolers)

        # =====================================================================
        # METRICS COMPILATION LAYER: FINALIZE KPI EXPORT
        # =====================================================================
        final_p_system = self._calculate_system_performance(cpu, gpu, use_case)
        total_cost = sum([
            float(cpu.get("price", 0)), float(gpu.get("price", 0)),
            float(motherboard.get("price", 0)), float(ram.get("price", 0)),
            float(psu.get("price", 0)), float(storage.get("price", 0)),
            float(case.get("price", 0)), float(cooler.get("price", 0))
        ])

        return {
            "strategy": strategy,
            "use_case": use_case,
            "cpu": cpu,
            "gpu": gpu,
            "motherboard": motherboard,
            "ram": ram,
            "psu": psu,
            "storage": storage,
            "case": case,
            "cooler": cooler,
            "metrics": {
                "p_system": final_p_system,
                "total_hardware_cost": round(total_cost, 2)
            }
        }

In [43]:
# ============================================
# LOAD NEXT-BEST-BUY DATA FROM LIVE GOOGLE SHEETS
# ============================================

CUSTOMER_NEXT_BUY_HISTORY_URL = "https://docs.google.com/spreadsheets/d/1vPUFL2UMarwzN3Nif40KSffenn8oJGdvtLLMbQWDaXU/export?format=csv"
ADDON_CATALOG_URL = "https://docs.google.com/spreadsheets/d/19Bfy9IIyhxIZzZR3LS4qUQj-KqCBaFSZx9Q22wngR6k/export?format=csv"


# ============================================
# AGENT 4.5 : NEXT-BEST-BUY & VALUE UPSELL AGENT
# ============================================

class NextBestBuyUpsellAgent:
    """
    Generates active sales recommendations after a PC build is selected.

    Data Sources:
    1. CUSTOMER_NEXT_BUY_HISTORY Google Sheet
       - Behaviour-based next-buy pattern

    2. ADDON_CATALOG Google Sheet
       - Add-on and upsell item catalog

    Output Sections:
    1. Frequently Bought Next
    2. Complete Your Setup
    3. Smart Budget Upsize
    """

    def __init__(
        self,
        history_url: str = CUSTOMER_NEXT_BUY_HISTORY_URL,
        addon_url: str = ADDON_CATALOG_URL
    ):
        self.history_url = history_url
        self.addon_url = addon_url

        self.history_df = self._load_google_sheet_safely(history_url, "CUSTOMER_NEXT_BUY_HISTORY")
        self.addon_df = self._load_google_sheet_safely(addon_url, "ADDON_CATALOG")

    def _load_google_sheet_safely(self, sheet_url: str, sheet_name: str):
        try:
            df = pd.read_csv(sheet_url)
            df.columns = df.columns.str.strip()
            print(f"✅ {sheet_name} loaded live from Google Sheets!")
            print(f"   Rows synced: {len(df)}")
            print(f"   Columns: {list(df.columns)}")
            return df

        except Exception as e:
            print(f"⚠️ Could not load {sheet_name} from Google Sheets: {e}")
            return pd.DataFrame()

    def _get_budget_range(self, budget: int) -> str:
        try:
            budget = int(budget)
        except Exception:
            budget = 4000

        if budget < 3000:
            return "2000-3000"
        elif budget < 5000:
            return "3000-5000"
        elif budget < 8000:
            return "5000-8000"
        else:
            return "8000+"

    def _priority_score(self, priority: str) -> int:
        priority = str(priority).lower()

        if priority == "high":
            return 3
        elif priority == "medium":
            return 2
        elif priority == "low":
            return 1

        return 0

    def _extract_component_names(self, components):
        return [str(c.get("name", "")).lower() for c in components]

    def _extract_component_types(self, components):
        return [str(c.get("type", "")).lower() for c in components]

    def _find_component_by_type(self, components, comp_type: str):
        for comp in components:
            if str(comp.get("type", "")).lower() == comp_type.lower():
                return comp
        return None

    # ------------------------------------------------------------
    # SECTION 1: FREQUENTLY BOUGHT NEXT
    # ------------------------------------------------------------

    def generate_frequently_bought_next(self, components, requirements, top_n: int = 5):
        if self.history_df.empty:
            return []

        df = self.history_df.copy()

        component_names = self._extract_component_names(components)
        component_types = self._extract_component_types(components)

        use_case = str(requirements.get("use_case", "general")).lower()
        budget_range = self._get_budget_range(requirements.get("budget", 4000))

        scored_rows = []

        for _, row in df.iterrows():
            score = 0

            trigger_item = str(row.get("trigger_item", "")).lower()
            trigger_category = str(row.get("trigger_category", "")).lower()
            row_use_case = str(row.get("use_case", "")).lower()
            row_budget_range = str(row.get("budget_range", "")).lower()
            accepted = str(row.get("accepted", "yes")).lower()

            if accepted != "yes":
                continue

            for name in component_names:
                if trigger_item and (trigger_item in name or name in trigger_item):
                    score += 5

            if trigger_category in component_types:
                score += 2

            if row_use_case == use_case:
                score += 2

            if row_budget_range == budget_range.lower():
                score += 1

            if score > 0:
                scored_rows.append({
                    "next_product": row.get("next_product", "-"),
                    "next_category": row.get("next_category", "-"),
                    "next_price": row.get("next_price", 0),
                    "reason": row.get("reason", "-"),
                    "score": score
                })

        if not scored_rows:
            return []

        scored_df = pd.DataFrame(scored_rows)

        grouped = (
            scored_df
            .groupby(["next_product", "next_category", "next_price"], as_index=False)
            .agg({
                "score": "sum",
                "reason": "first"
            })
            .sort_values(by="score", ascending=False)
            .head(top_n)
        )

        max_score = grouped["score"].max() if len(grouped) > 0 else 1

        recommendations = []

        for rank, (_, row) in enumerate(grouped.iterrows(), start=1):
            confidence = round((row["score"] / max_score) * 100, 1)

            recommendations.append({
                "rank": rank,
                "recommendation": row["next_product"],
                "category": row["next_category"],
                "price": int(row["next_price"]) if pd.notna(row["next_price"]) else 0,
                "similarity_signal": f"{confidence}%",
                "reason": row["reason"]
            })

        return recommendations

    # ------------------------------------------------------------
    # SECTION 2: COMPLETE YOUR SETUP
    # ------------------------------------------------------------

    def generate_complete_your_setup(self, components, requirements, top_n: int = 5):
        """
        Recommends add-ons and accessories based on the selected build.
        Uses ADDON_CATALOG Google Sheet data.
        """
        if self.addon_df.empty:
            return []

        df = self.addon_df.copy()

        use_case = str(requirements.get("use_case", "general")).lower()

        # Determine column names (flexible for different sheet structures)
        name_col = "addon_name" if "addon_name" in df.columns else "name"
        category_col = "addon_category" if "addon_category" in df.columns else "category"
        use_case_col = "suitable_use_case" if "suitable_use_case" in df.columns else "use_case"
        price_col = "price"
        priority_col = "priority"
        reason_col = "reason"

        if name_col not in df.columns:
            return []

        # Filter by use case
        if use_case_col in df.columns:
            filtered = df[
                df[use_case_col].astype(str).str.lower().isin([use_case, "general", "all"])
            ].copy()
        else:
            filtered = df.copy()

        if filtered.empty:
            filtered = df.copy()

        # Add priority score
        if priority_col in filtered.columns:
            filtered["priority_score"] = filtered[priority_col].apply(self._priority_score)
        else:
            filtered["priority_score"] = 1

        # Add price column if missing
        if price_col not in filtered.columns:
            filtered[price_col] = 0

        # Sort by priority then price
        filtered = filtered.sort_values(
            by=["priority_score", price_col],
            ascending=[False, True]
        ).head(top_n)

        recommendations = []

        for rank, (_, row) in enumerate(filtered.iterrows(), start=1):
            recommendations.append({
                "rank": rank,
                "recommendation": row.get(name_col, "-"),
                "category": row.get(category_col, "-"),
                "price": int(row.get(price_col, 0)) if pd.notna(row.get(price_col, 0)) else 0,
                "priority": row.get(priority_col, "Medium"),
                "reason": row.get(reason_col, "Recommended to complete your setup.")
            })

        return recommendations

    # ------------------------------------------------------------
    # SECTION 3: SMART BUDGET UPSIZE
    # ------------------------------------------------------------

    def generate_smart_budget_upsize(self, components, requirements, top_n: int = 5):
        """
        Finds better component upgrades within budget.
        Searches PRODUCT_CATALOG for higher-tier alternatives.
        """
        recommendations = []

        use_case = str(requirements.get("use_case", "general")).lower()
        budget = requirements.get("budget", 4000)

        try:
            budget = float(budget)
        except Exception:
            budget = 4000

        current_total = sum(float(c.get("price", 0)) for c in components)
        remaining_budget = max(0, budget - current_total)

        # Category mapping
        category_map = {
            "CPUs": "CPUs", "CPU": "CPUs",
            "GPUs": "GPUs", "GPU": "GPUs",
            "Motherboards": "Motherboards", "Motherboard": "Motherboards",
            "RAM": "RAM",
            "Storage": "Storage",
            "PSU": "PSU",
            "Cases": "Cases", "Case": "Cases",
            "Coolers": "Coolers", "Cooler": "Coolers"
        }

        def get_catalog_category(comp_type):
            return category_map.get(str(comp_type), str(comp_type))

        def get_upgrade_score(current_item, better_item, category):
            current_price = float(current_item.get("price", 0))
            better_price = float(better_item.get("price", 0))
            extra_cost = better_price - current_price

            if extra_cost <= 0:
                return -999

            score = 0

            # Affordability score
            if extra_cost <= remaining_budget:
                score += 25
            elif extra_cost <= remaining_budget + 300:
                score += 12
            else:
                score -= 15

            # Category importance by use case
            if use_case == "gaming":
                if category == "GPUs":
                    score += 35
                elif category == "CPUs":
                    score += 22
                elif category == "RAM":
                    score += 18
                elif category == "Storage":
                    score += 16
                elif category == "PSU":
                    score += 15
                elif category == "Coolers":
                    score += 10
            elif use_case == "workstation":
                if category == "RAM":
                    score += 35
                elif category == "CPUs":
                    score += 28
                elif category == "Storage":
                    score += 26
                elif category == "GPUs":
                    score += 18
                elif category == "PSU":
                    score += 14
                elif category == "Coolers":
                    score += 10
            else:
                if category in ["RAM", "Storage", "PSU"]:
                    score += 22
                elif category in ["CPUs", "GPUs"]:
                    score += 15

            # Performance improvement
            current_perf = float(current_item.get("gaming_performance", 0))
            better_perf = float(better_item.get("gaming_performance", 0))
            if better_perf > current_perf:
                score += min(30, better_perf - current_perf)

            # Capacity improvement (RAM/Storage)
            current_capacity = float(current_item.get("capacity", 0) or 0)
            better_capacity = float(better_item.get("capacity", 0) or 0)
            if category in ["RAM", "Storage"] and better_capacity > current_capacity:
                score += min(30, (better_capacity - current_capacity) * 8)

            # Wattage improvement (PSU)
            current_wattage = float(current_item.get("wattage", 0) or 0)
            better_wattage = float(better_item.get("wattage", 0) or 0)
            if category == "PSU" and better_wattage > current_wattage:
                score += min(25, (better_wattage - current_wattage) / 20)

            # Value penalty if too expensive
            if extra_cost > budget * 0.25:
                score -= 20
            elif extra_cost > budget * 0.15:
                score -= 10

            return score

        def build_benefit_text(current_item, better_item, category):
            current_name = current_item.get("name", "current item")
            better_name = better_item.get("name", "better option")

            benefits = {
                "GPUs": f"Moving from {current_name} to {better_name} can improve gaming smoothness, graphics quality, and long-term game readiness.",
                "CPUs": f"Moving from {current_name} to {better_name} can improve multitasking, processing speed, and overall system responsiveness.",
                "RAM": f"Moving from {current_name} to {better_name} can improve multitasking, heavier applications, and future software comfort.",
                "Storage": f"Moving from {current_name} to {better_name} can provide more space and faster loading for games, files, and applications.",
                "PSU": f"Moving from {current_name} to {better_name} can improve power stability and give better headroom for future upgrades.",
                "Coolers": f"Moving from {current_name} to {better_name} can reduce temperature and noise, especially under heavier workloads.",
                "Cases": f"Moving from {current_name} to {better_name} can improve airflow, cable space, and overall build presentation."
            }
            return benefits.get(category, f"Moving from {current_name} to {better_name} gives a stronger long-term configuration.")

        # Search for upgrades
        for current_item in components:
            current_type = current_item.get("type", "")
            category = get_catalog_category(current_type)

            if category not in PRODUCT_CATALOG:
                continue

            current_price = float(current_item.get("price", 0))
            catalog_items = PRODUCT_CATALOG.get(category, [])

            upgrade_candidates = [
                item for item in catalog_items
                if item.get("stock", 0) > 0
                and float(item.get("price", 0)) > current_price
                and item.get("name") != current_item.get("name")
            ]

            # Compatibility filters
            if category == "Motherboards":
                current_socket = current_item.get("socket")
                upgrade_candidates = [item for item in upgrade_candidates if item.get("socket") == current_socket]

            if category == "RAM":
                current_ram_type = str(current_item.get("ram_type", "")).upper()
                upgrade_candidates = [item for item in upgrade_candidates if str(item.get("ram_type", "")).upper() == current_ram_type]

            if not upgrade_candidates:
                continue

            scored_candidates = []
            for candidate in upgrade_candidates:
                score = get_upgrade_score(current_item, candidate, category)
                if score > 0:
                    scored_candidates.append({"candidate": candidate, "score": score})

            if not scored_candidates:
                continue

            best_candidate = sorted(scored_candidates, key=lambda x: x["score"], reverse=True)[0]["candidate"]
            extra_cost = float(best_candidate.get("price", 0)) - current_price

            if extra_cost <= 0:
                continue

            priority = "High" if extra_cost <= remaining_budget or category in ["RAM", "Storage", "PSU"] else "Medium"

            recommendations.append({
                "rank": 0,
                "current_item": current_item.get("name", "-"),
                "recommended_upgrade": best_candidate.get("name", "-"),
                "extra_cost": int(extra_cost),
                "priority": priority,
                "benefit": build_benefit_text(current_item, best_candidate, category),
                "category": category,
                "score": round(get_upgrade_score(current_item, best_candidate, category), 2)
            })

        # Sort by score and limit
        recommendations = sorted(recommendations, key=lambda x: x.get("score", 0), reverse=True)[:top_n]

        for i, rec in enumerate(recommendations, start=1):
            rec["rank"] = i
            rec.pop("score", None)

        return recommendations

    # ------------------------------------------------------------
    # MAIN OUTPUT METHOD
    # ------------------------------------------------------------

    def generate_recommendations(self, components, requirements, optimization):
        """
        Main method that combines all three recommendation types.
        Returns a complete upsell package.
        """
        frequently_bought_next = self.generate_frequently_bought_next(
            components,
            requirements
        )

        complete_your_setup = self.generate_complete_your_setup(
            components,
            requirements
        )

        smart_budget_upsize = self.generate_smart_budget_upsize(
            components,
            requirements
        )

        return {
            "frequently_bought_next": frequently_bought_next,
            "complete_your_setup": complete_your_setup,
            "smart_budget_upsize": smart_budget_upsize
        }

In [44]:
# ============================================
# AGENT 5: COMMERCIAL QUOTATION AGENT (CASE-INSENSITIVE)
# ============================================

class QuotationAgent:
    """Compiles official physical purchase order bills or alternative suggestions"""

    def generate_quotation(self, components: List[Dict], requirements: Dict, compatibility: Dict, optimization: Dict) -> str:
        total = sum(c.get("price", 0) for c in components)
        location = requirements.get("location", "Unknown location").strip()
        delivery_available = requirements.get("delivery_available", False)

        global DELIVERY_LOCATIONS

        # FIXED: Normalize the lookup key to be case-insensitive to catch 'Petaling Jaya', 'petaling jaya', etc.
        matched_location_key = None
        if 'DELIVERY_LOCATIONS' in globals() and DELIVERY_LOCATIONS:
            for key in DELIVERY_LOCATIONS.keys():
                if key.lower() == location.lower():
                    matched_location_key = key
                    break

        # If a case-insensitive match was found in our dictionary, force delivery_available to True
        if matched_location_key:
            delivery_available = True
            delivery = DELIVERY_LOCATIONS[matched_location_key]
            shipping_cost = delivery.get('delivery_fee', 0)
            delivery_days = delivery.get('estimated_days', '3 business days')
            total_price = total + shipping_cost
            show_prices = True
            delivery_status = f"✓ Delivery Verified to {matched_location_key}"
            delivery_line = f"Estimated Delivery Transit window: {delivery_days}"
            shipping_line = f"Shipping Logistic Fee (to {matched_location_key}){' ' * 15}RM{shipping_cost:>8,}"
        else:
            shipping_cost = 0
            delivery_status = f"✗ LOGISTIC DELIVERY COVERAGE NOT AVAILABLE TO: {location}"
            delivery_line = "We are unable to dispatch freight packages to this area. Price index hidden."
            shipping_line = f"Shipping Route Status{' ' * 30}SERVICE COVERAGE FAULT"
            total_price = None
            show_prices = False

        if show_prices:
            quote = f"{'='*60}\n{' '*20}OFFICIAL COMMERCIAL QUOTATION\n{'='*60}\n\n"
            quote += f"CLIENT LOCATION: {matched_location_key}\nORDER DATE STAMP: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
            quote += f"\nLOGISTICS NETWORK BREAKDOWN:\n{'-'*60}\n{delivery_status}\n{delivery_line}\n\n"
            quote += f"SPECIFIED PROCUREMENT ITEMIZATION:\n{'-'*60}\n"
            for c in components:
                quote += f"- [{c.get('type','PART'):<12}] {c.get('name','Model'):<32} RM{c.get('price',0):>6,}\n"
            quote += f"\n{'-'*60}\nSUBTOTAL BASE PACKAGE COST:{' '*21}RM{total:>8,}\n{shipping_line}\n{'-'*60}\n"
            quote += f"TOTAL COMMERCIAL BILL COST:{' '*21}RM{total_price:>8,}\n{'='*60}"
        else:
            quote = f"{'='*60}\n{' '*20}COMPILATION CONFIGURATION SUGGESTION\n{'='*60}\n\n"
            quote += f"TARGET AREA FOCUS: {location} (Logistical Constraint Rule Triggered)\n"
            quote += f"\nLOGISTICS NETWORK BREAKDOWN:\n{'-'*60}\n{delivery_status}\n{delivery_line}\n\n"
            quote += f"COMPATIBLE HARDWARE DISCOVERED:\n{'-'*60}\n"
            for c in components:
                quote += f"• [{c.get('type','PART')}] {c.get('name','Model')} \n"
            quote += f"\n{'='*60}\n⚠️ NOTICE: Due to destination boundary exceptions, please consult regional tech channels (e.g., Low Yat Plaza, Shopee Malaysia) to purchase these components locally."

        return quote

In [ ]:
# ============================================================
# 4. AGENT 6: EXPLANATION AGENT (STRATEGY-AWARE - VARIABLE FIX)
# ============================================================

class ExplanationAgent:
    """Explains why each component was selected - LLM-POWERED for natural language"""

    def __init__(self):
        pass

    def generate_explanation(self, components: List[Dict], requirements: Dict, strategy_name: str) -> str:
        """Generate strategy-focused detailed explanation using Gemini with dynamic variable fallbacks"""

        # Extract variables outside the multi-line block to enforce clean interpolation
        req_budget = requirements.get('budget', 4000)
        req_location = requirements.get('location', 'Unknown')
        req_use_case = requirements.get('use_case', 'Gaming')
        components_json = json.dumps(components, indent=2)

        prompt = f"""
        You are an expert PC builder and IT data analyst. Explain why each of these components was selected
        for the user's build, specifically focusing on the user's chosen optimization strategy: {strategy_name}.

        User Requirements Profile:
        - Target Budget Threshold: RM {req_budget}
        - Delivery Location: {req_location}
        - Target Use Case: {req_use_case}
        - Strategic Optimization Focus: {strategy_name}

        Selected Component Configuration Matrix:
        {components_json}

        Provide a comprehensive, persuasive breakdown for the buyer covering:
        1. An introductory statement explaining why this system configuration perfectly matches a "{strategy_name}" focus.
        2. Detailed analysis for each component: Why this exact model fits the theme, its price-to-performance contribution, and how it avoids bottlenecking.
        3. A structural compatibility summary confirming electrical (PSU) and physical layout alignment.

        Format your response in beautiful, scannable Markdown with clean headings. Keep the tone professional, authoritative, yet friendly.
        """

        try:
            # Check for standard unified google-genai client object
            if 'ai_client' in globals():
                response = ai_client.models.generate_content(model='gemini-3.5-flash', contents=prompt)
                return response.text
            # Check for alternative 'client' naming pattern
            elif 'client' in globals():
                response = client.models.generate_content(model='gemini-3.5-flash', contents=prompt)
                return response.text
            # Check for legacy google-generativeai module pattern
            elif 'genai' in globals():
                model = genai.GenerativeModel('gemini-3.5-flash')
                response = model.generate_content(prompt)
                return response.text
            else:
                raise NameError("No valid Gemini API client discovered in global workspace environment.")

        except Exception as e:
            print(f"⚠️ Gemini execution bypassed to professional fallback layer. Log: {e}")
            return self._generate_fallback_explanation(components, requirements, strategy_name)

    def _generate_fallback_explanation(self, components: List[Dict], requirements: Dict, strategy_name: str) -> str:
        """Dynamic, high-tier fallback engine that maps accurately to exact catalog category keys"""
        use_case = str(requirements.get('use_case', 'Gaming')).title()

        md = f"## 🚀 System Architecture Analysis: {strategy_name} Profile\n\n"
        md += f"We have successfully compiled and cross-validated your custom hardware configuration targeted specifically at **{use_case}** performance vectors. "
        md += f"This deployment carefully enforces your **RM {requirements.get('budget', 4000):,}** cost boundary while pulling active stock items directly from our inventory channels.\n\n"

        md += "### 📦 Selected Component Role Allocations:\n"

        for comp in components:
            c_type = str(comp.get('type', '')).upper().strip()
            c_name = comp.get('name', 'Generic Part')
            c_price = comp.get('price', 0)

            md += f"#### • [{comp.get('type', 'PART')}] {c_name} (RM {c_price:,})\n"

            if "CPU" in c_type:
                md += f"  Selected as the foundational computational engine for this build. Its thread allocation and core design ensure seamless execution across background processing pipelines while offering zero bottleneck constraints to your graphics matrix.\n\n"
            elif "GPU" in c_type:
                md += f"  Serving as the primary visual performance driver. Chosen to deliver high framerates and stable rendering passes under heavy data loads, striking the absolute best balance matching our strategic framework.\n\n"
            elif "MOTHERBOARD" in c_type:
                md += f"  Acts as the central system backbone. Enforces absolute circuit compatibility with your socket array and memory controller chips to guarantee long-term system stability.\n\n"
            elif "RAM" in c_type:
                md += f"  Provides high-bandwidth transient memory maps to eliminate frame stuttering and data-swapping delays during multi-threaded operation cycles.\n\n"
            elif "STORAGE" in c_type:
                md += f"  High-speed solid-state interface selected to drop system boot parameters and application loading sequences down to near-zero latency barriers.\n\n"
            elif "PSU" in c_type:
                md += f"  Enforces clean power delivery across all computing nodes, offering plenty of operational wattage safety overhead to handle peak electricity draws safely.\n\n"
            elif "CASE" in c_type:
                md += f"  Provides a durable physical enclosure with optimized structural airflow layout properties to maintain healthy ambient component temperatures.\n\n"
            elif "COOLER" in c_type:
                md += f"  Maintains thermal equilibrium across critical processor architectures, preventing thermal throttling limits from cutting frame processing speeds.\n\n"
            else:
                md += f"  Fully certified deployment accessory. Verified for operational utility parameters matching your selected architectural system space.\n\n"

        md += "---\n### 🛡️ System Engineering Balance Check\n"
        md += "Our pipeline verified that your component values match perfectly across your entire processing engine. Memory latencies, power distribution lines, and computational tier balancing calculations all indicate a highly responsive, stable build profile ready for immediate local procurement."

        return md
# ============================================================
# 5. MAIN SYSTEM ORCHESTRATOR & CORES
# ============================================================

class AutonomousSalesAgent:
    """
    Main orchestrator for the Autonomous PC Sales Engineer pipeline.

    Pipeline State Flow:
    1. Requirement Agent
       -> self.state["requirements"]

    2. Search Agent
       -> self.state["components"]

    3. Compatibility Agent
       -> self.state["compatibility"]

    4. Optimizer Agent
       -> self.state["optimization"]

    5. Next-Best-Buy & Value Options Agent
       -> self.state["upsell_recommendations"]

    6. Quotation Agent
       -> called later inside process_sales_pipeline()
       -> uses selected strategy build

    7. Explanation Agent
       -> called later inside process_sales_pipeline()
       -> uses selected strategy build + upsell recommendations
    """

    def __init__(self):
        self.requirement_agent = RequirementAgent()
        self.search_agent = SearchAgent()
        self.compatibility_agent = CompatibilityAgent()
        self.optimizer_agent = OptimizerAgent()
        self.upsell_agent = NextBestBuyUpsellAgent()
        self.quotation_agent = QuotationAgent()
        self.explanation_agent = ExplanationAgent()
        self.state = {}

    def run(self, user_input: str) -> Dict:
        print("\n" + "=" * 60)
        print("🤖 AUTONOMOUS SALES ENGINEER AGENT ACTIVE")
        print("=" * 60)

        # Reset state for every new customer request
        self.state = {}

        # =====================================================
        # AGENT 1: REQUIREMENT AGENT
        # =====================================================
        print("\n📋 AGENT 1: REQUIREMENT AGENT")
        self.state["requirements"] = self.requirement_agent.extract_requirements(
            user_input
        )

        # =====================================================
        # AGENT 2: SEARCH AGENT
        # =====================================================
        print("\n🔍 AGENT 2: SEARCH AGENT")
        self.state["components"] = self.search_agent.search_components(
            self.state["requirements"]
        )

        # =====================================================
        # AGENT 3: COMPATIBILITY AGENT
        # =====================================================
        print("\n🛡️ AGENT 3: COMPATIBILITY AGENT")
        self.state["compatibility"] = self.compatibility_agent.check_compatibility(
            self.state["components"]
        )

        # =====================================================
        # AGENT 4: OPTIMIZER AGENT
        # =====================================================
        print("\n⚡ AGENT 4: OPTIMIZER AGENT")
        self.state["optimization"] = self.optimizer_agent.optimize_build(
            self.state["components"],
            self.state["requirements"].get("budget", 4000)
        )

        # =====================================================
        # AGENT 5: NEXT-BEST-BUY & VALUE OPTIONS AGENT
        # =====================================================
        print("\n💡 AGENT 5: NEXT-BEST-BUY & VALUE OPTIONS AGENT")
        self.state["upsell_recommendations"] = self.upsell_agent.generate_recommendations(
            self.state["components"],
            self.state["requirements"],
            self.state["optimization"]
        )

        print("\n✅ PIPELINE STATE GENERATED SUCCESSFULLY")
        print("=" * 60)

        return self.state


# Initialize Engine Global Variable
AGENT = AutonomousSalesAgent()
# ============================================================
# 6. INTERACTIVE PIPELINE RENDERING DATA METRIC HELPERS
# ============================================================

def _safe_money(value):
    try:
        return float(value)
    except Exception:
        return 0.0


def _build_strategy_summary(strategies):
    rows = []

    if not strategies:
        return pd.DataFrame(
            columns=[
                "Setup Option",
                "Total Price (RM)",
                "Estimated Speed Rating",
                "Value Efficiency"
            ]
        )

    for strategy_name, info in strategies.items():
        rows.append({
            "Setup Option": strategy_name.replace("_", " ").title(),
            "Total Price (RM)": f"RM {info.get('total_cost', 0):,}",
            "Estimated Speed Rating": f"{round(info.get('performance_score', 0), 1)} / 500",
            "Value Efficiency": f"{round(info.get('value_ratio', 0), 4)} points/RM"
        })

    return pd.DataFrame(rows)


def _build_component_matrix(strategies):
    rows = []

    if not strategies:
        return pd.DataFrame(
            columns=[
                "Build Profile",
                "Part Category",
                "Component Model",
                "Price",
                "Warehouse Status"
            ]
        )

    for strategy_name, info in strategies.items():
        display_strategy = strategy_name.replace("_", " ").title()

        for item in info.get("components", []):
            stock_num = item.get("stock", 0)

            stock_str = (
                "✅ In Stock"
                if stock_num > 10
                else ("⚠️ Low Stock" if stock_num > 0 else "❌ Reserved")
            )

            rows.append({
                "Build Profile": display_strategy,
                "Part Category": item.get("type", "-"),
                "Component Model": item.get("name", "-"),
                "Price": f"RM {item.get('price', 0):,}",
                "Warehouse Status": stock_str
            })

    return pd.DataFrame(rows)


def _build_compatibility_markdown(compat):
    status = compat.get("status", "Unknown")
    messages = compat.get("messages", [])

    md = f"### System Health Status: **Verified {status}**\n\n"

    if messages:
        for msg in messages:
            md += f"✅ {msg}\n"
    else:
        md += "✅ Component compatibility validated across selected system parts.\n"

    return md


def _build_requirement_markdown(requirements):
    if not requirements:
        return "No active customer profile analyzed yet."

    return f"""
<div class="profile-summary">
    <div class="profile-item">
        <span class="profile-label">Budget</span>
        <span class="profile-value">RM {requirements.get("budget", "-")}</span>
    </div>
    <div class="profile-item">
        <span class="profile-label">Delivery Target</span>
        <span class="profile-value">{requirements.get("location", "-")}</span>
    </div>
    <div class="profile-item">
        <span class="profile-label">Main Use Case</span>
        <span class="profile-value">{str(requirements.get("use_case", "-")).title()}</span>
    </div>
</div>
"""

def _calculate_selected_build_total(components, requirements):
    """
    Calculates the visible final selected build total based on chosen components.
    This keeps KPI budget space consistent with the quotation section.
    """

    component_total = 0

    for item in components:
        try:
            component_total += float(item.get("price", 0))
        except Exception:
            component_total += 0

    shipping_fee = 0

    for key in ["delivery_fee", "shipping_fee", "delivery_cost"]:
        if key in requirements:
            try:
                shipping_fee = float(requirements.get(key, 0))
                break
            except Exception:
                shipping_fee = 0

    return component_total + shipping_fee
# ============================================================
# PREMIUM RECOMMENDATION CARD HELPERS
# ============================================================

def _get_first_item(items):
    if items and len(items) > 0:
        return items[0]
    return {}


def _format_price(value, prefix="RM"):
    try:
        return f"{prefix} {int(value):,}"
    except Exception:
        return f"{prefix} 0"


def _build_value_recommendation_cards(upsell_data):
    """
    Converts the 3 upsell recommendation groups into premium customer-facing cards.
    This avoids dumping long tables into the main dashboard while preserving the full details elsewhere.
    """

    you_may_need = _get_first_item(upsell_data.get("frequently_bought_next", []))
    complete_setup = _get_first_item(upsell_data.get("complete_your_setup", []))
    better_value = _get_first_item(upsell_data.get("smart_budget_upsize", []))

    # -----------------------------
    # CARD 1: YOU MAY ALSO NEED
    # -----------------------------
    if you_may_need:
        card_1_title = you_may_need.get("recommendation", "Suggested Item")
        card_1_price = you_may_need.get("price", 0)
        card_1_reason = you_may_need.get("reason", "Often selected by similar customers.")
        card_1_signal = you_may_need.get("similarity_signal", "Strong match")
    else:
        card_1_title = "No immediate extra item needed"
        card_1_price = 0
        card_1_reason = "Your selected setup already covers the core requirement."
        card_1_signal = "No strong pattern detected"

    # -----------------------------
    # CARD 2: COMPLETE YOUR SETUP
    # -----------------------------
    if complete_setup:
        card_2_title = complete_setup.get("recommendation", "Suggested Add-on")
        card_2_price = complete_setup.get("price", 0)
        card_2_reason = complete_setup.get("reason", "Helps improve the full setup experience.")
        card_2_priority = complete_setup.get("priority", "Recommended")
    else:
        card_2_title = "Setup already feels complete"
        card_2_price = 0
        card_2_reason = "No essential add-on is required for this configuration."
        card_2_priority = "Optional"

    # -----------------------------
    # CARD 3: BETTER VALUE OPTION
    # -----------------------------
    if better_value:
        card_3_title = better_value.get("recommended_upgrade", "Better Option")
        card_3_price = better_value.get("extra_cost", 0)
        card_3_reason = better_value.get("benefit", "Provides stronger long-term value.")
        card_3_current = better_value.get("current_item", "Current choice")
    else:
        card_3_title = "No value upgrade needed"
        card_3_price = 0
        card_3_reason = "The selected setup already gives a balanced value profile."
        card_3_current = "Current setup"

    return f"""
<div class="recommendation-grid">

    <div class="value-card blue-card">
        <div class="card-eyebrow">💡 You May Also Need</div>
        <div class="card-title">{card_1_title}</div>
        <div class="card-price">{_format_price(card_1_price, "RM")}</div>
        <div class="card-body">{card_1_reason}</div>
        <div class="card-tag">Customer Match: {card_1_signal}</div>
    </div>

    <div class="value-card green-card">
        <div class="card-eyebrow">🧩 Make Your Setup Complete</div>
        <div class="card-title">{card_2_title}</div>
        <div class="card-price">{_format_price(card_2_price, "RM")}</div>
        <div class="card-body">{card_2_reason}</div>
        <div class="card-tag">Priority: {card_2_priority}</div>
    </div>

    <div class="value-card amber-card">
        <div class="card-eyebrow">📈 Better Value Option</div>
        <div class="card-title">{card_3_title}</div>
        <div class="card-price"> Top up RM {int(card_3_price):,}</div>
        <div class="card-body">{card_3_reason}</div>
        <div class="card-tag">Current: {card_3_current}</div>
    </div>

</div>
"""


def _build_recommendation_detail_df(upsell_data):
    """
    Builds one compact detail table for optional review.
    This keeps every recommendation available without overwhelming the main screen.
    """

    rows = []

    for item in upsell_data.get("frequently_bought_next", []):
        rows.append({
            "Section": "You May Also Need",
            "Suggestion": item.get("recommendation", "-"),
            "Price": f"RM {item.get('price', 0):,}",
            "Why": item.get("reason", "-")
        })

    for item in upsell_data.get("complete_your_setup", []):
        rows.append({
            "Section": "Make Your Setup Complete",
            "Suggestion": item.get("recommendation", "-"),
            "Price": f"RM {item.get('price', 0):,}",
            "Why": item.get("reason", "-")
        })

    for item in upsell_data.get("smart_budget_upsize", []):
        rows.append({
            "Section": "Better Value Option",
            "Suggestion": item.get("recommended_upgrade", "-"),
            "Price": f"+RM {item.get('extra_cost', 0):,}",
            "Why": item.get("benefit", "-")
        })

    if not rows:
        return pd.DataFrame(
            columns=[
                "Section",
                "Suggestion",
                "Price",
                "Why"
            ]
        )

    return pd.DataFrame(rows)


# ============================================================
# 7. INTERACTIVE PIPELINE CORE EXECUTION LINK
# ============================================================

def process_sales_pipeline(user_query, budget_input, strategy_choice):
    empty_df_strategy = pd.DataFrame(
        columns=[
            "Setup Option",
            "Total Price (RM)",
            "Estimated Speed Rating",
            "Value Efficiency"
        ]
    )

    empty_df_components = pd.DataFrame(
        columns=[
            "Build Profile",
            "Part Category",
            "Component Model",
            "Price",
            "Warehouse Status"
        ]
    )

    empty_df_recommendation_details = pd.DataFrame(
        columns=[
            "Section",
            "Suggestion",
            "Price",
            "Why"
        ]
    )

    if not user_query or not user_query.strip():
        return (
            "Please provide your preferences or target usage details above.",
            0,
            0,
            0,
            0,
            empty_df_strategy,
            empty_df_components,
            "<div class='empty-card'>Recommendations will appear after the system builds your setup.</div>",
            empty_df_recommendation_details,
            "Awaiting input.",
            "Awaiting input.",
            "Awaiting input."
        )

    try:
        full_input = f"{user_query} with a budget of RM{budget_input}"
        state = AGENT.run(full_input)

    except Exception as e:
        error_message = f"🚨 System Check Alert: {str(e)}"

        return (
            error_message,
            0,
            0,
            0,
            0,
            empty_df_strategy,
            empty_df_components,
            f"<div class='empty-card'>{error_message}</div>",
            empty_df_recommendation_details,
            error_message,
            error_message,
            error_message
        )

    requirements = state.get("requirements", {})
    optimization = state.get("optimization", {})
    strategies = optimization.get("strategies", {})
    compatibility = state.get("compatibility", {})

    strategy_mapping = {
        "🚀 Max Performance": "max_performance",
        "⚖️ Optimal Value": "optimal_value",
        "📉 Budget Saver": "budget_saver"
    }

    selected_key = strategy_mapping.get(strategy_choice, "optimal_value")

    if strategies and selected_key in strategies:
        target_build = strategies[selected_key]
        chosen_components = target_build.get("components", state.get("components", []))
        display_cost = target_build.get("total_cost", 0)
        display_perf = target_build.get("performance_score", 0)
    else:
        chosen_components = state.get("components", [])
        display_cost = optimization.get("optimized_total", 0)
        display_perf = optimization.get("performance_score", 0)
    display_cost = _calculate_selected_build_total(
        chosen_components,
        requirements
    )
    # Regenerate recommendations based on the selected strategy build
    state["upsell_recommendations"] = AGENT.upsell_agent.generate_recommendations(
        chosen_components,
        requirements,
        optimization
    )

    quotation_text = AGENT.quotation_agent.generate_quotation(
        chosen_components,
        requirements,
        compatibility,
        {
            "within_budget": display_cost <= budget_input,
            "performance_score": display_perf,
            "optimized_total": display_cost
        }
    )

    explanation_text = AGENT.explanation_agent.generate_explanation(
        chosen_components,
        requirements,
        strategy_choice
    )

    req_markdown = _build_requirement_markdown(requirements)

    original_total = _safe_money(optimization.get("original_total", 0))
    savings = max(0.0, float(budget_input) - float(display_cost))

    strategy_summary_df = _build_strategy_summary(strategies)
    component_matrix_df = _build_component_matrix(strategies)

    upsell_data = state.get("upsell_recommendations", {})

    recommendation_cards_html = _build_value_recommendation_cards(upsell_data)
    recommendation_details_df = _build_recommendation_detail_df(upsell_data)

    compatibility_markdown = _build_compatibility_markdown(compatibility)

    return (
        req_markdown,
        original_total,
        display_cost,
        savings,
        display_perf,
        strategy_summary_df,
        component_matrix_df,
        recommendation_cards_html,
        recommendation_details_df,
        compatibility_markdown,
        quotation_text,
        explanation_text
    )

# ============================================================
# 8. HUMAN-CENTRIC GRADIO INTERFACE LAYOUT (PREMIUM TABBED VERSION + VISIBLE EXTRAS)
# ============================================================

CUSTOM_CSS = """
#main-header {
    text-align: left;
    padding: 24px 30px;
    border-radius: 18px;
    background:
        radial-gradient(circle at top left, rgba(59,130,246,0.35), transparent 30%),
        linear-gradient(135deg, #020617 0%, #0F172A 55%, #1E293B 100%);
    color: white;
    margin-bottom: 22px;
    box-shadow: 0 18px 40px rgba(15, 23, 42, 0.20);
}

#main-header h1 {
    margin: 0 0 6px 0;
    font-size: 28px;
    font-weight: 800;
    color: #FAFAFA;
    letter-spacing: -0.7px;
}

#main-header p {
    color: #CBD5E1;
    font-size: 14px;
    margin: 0;
    line-height: 1.5;
}

.input-card {
    padding: 18px;
    border-radius: 18px;
    background: #FFFFFF;
    border: 1px solid #E2E8F0;
    box-shadow: 0 8px 22px rgba(15, 23, 42, 0.05);
}

.hero-section {
    padding: 18px 20px;
    border-radius: 18px;
    background: linear-gradient(135deg, #F8FAFC 0%, #FFFFFF 70%);
    border: 1px solid #E2E8F0;
    margin-bottom: 18px;
}

.extras-section {
    padding: 18px 20px;
    border-radius: 18px;
    background: linear-gradient(135deg, #F0FDF4 0%, #FFFFFF 72%);
    border: 1px solid #BBF7D0;
    margin-bottom: 18px;
}

.hero-title {
    font-size: 24px;
    font-weight: 850;
    color: #1E3A8A;
    margin-bottom: 6px;
    letter-spacing: -0.4px;
}

.extras-title {
    font-size: 23px;
    font-weight: 850;
    color: #065F46;
    margin-bottom: 6px;
    letter-spacing: -0.4px;
}

.hero-subtitle {
    color: #64748B;
    font-size: 13.5px;
    margin-bottom: 12px;
}

.clean-invoice textarea {
    font-family: "Consolas", monospace !important;
    font-size: 13px !important;
    background-color: #F8FAFC !important;
    color: #0F172A !important;
    border: 1px solid #E2E8F0 !important;
    border-radius: 12px !important;
}

.scroll-box {
    max-height: 520px;
    overflow-y: auto;
    padding: 8px;
}

.section-note {
    color: #64748B;
    font-size: 13px;
    line-height: 1.45;
    margin-top: -3px;
    margin-bottom: 12px;
}

.value-card {
    border-radius: 18px;
    padding: 16px 17px;
    margin-bottom: 14px;
    border: 1px solid rgba(148, 163, 184, 0.35);
    background: white;
    box-shadow: 0 10px 28px rgba(15, 23, 42, 0.07);
}

.blue-card {
    background: linear-gradient(135deg, #EFF6FF 0%, #FFFFFF 70%);
    border-left: 5px solid #2563EB;
}

.green-card {
    background: linear-gradient(135deg, #ECFDF5 0%, #FFFFFF 70%);
    border-left: 5px solid #059669;
}

.amber-card {
    background: linear-gradient(135deg, #FFFBEB 0%, #FFFFFF 70%);
    border-left: 5px solid #D97706;
}

.card-eyebrow {
    font-size: 12px;
    font-weight: 800;
    text-transform: uppercase;
    letter-spacing: 0.04em;
    color: #475569;
    margin-bottom: 6px;
}

.card-title {
    font-size: 18px;
    font-weight: 850;
    color: #0F172A;
    line-height: 1.2;
    margin-bottom: 4px;
}

.card-price {
    font-size: 22px;
    font-weight: 900;
    color: #111827;
    margin: 8px 0;
}

.card-body {
    color: #334155;
    font-size: 13.5px;
    line-height: 1.45;
    margin-bottom: 10px;
}

.card-tag {
    display: inline-block;
    padding: 5px 9px;
    border-radius: 999px;
    background: rgba(15, 23, 42, 0.07);
    color: #334155;
    font-size: 12px;
    font-weight: 700;
}

.empty-card {
    border-radius: 16px;
    padding: 18px;
    background: #F8FAFC;
    color: #64748B;
    border: 1px dashed #CBD5E1;
}

.profile-summary {
    display: grid;
    gap: 10px;
}

.profile-item {
    padding: 12px 13px;
    border-radius: 14px;
    background: #F8FAFC;
    border: 1px solid #E2E8F0;
}

.profile-label {
    display: block;
    color: #64748B;
    font-size: 12px;
    font-weight: 700;
    margin-bottom: 3px;
}

.profile-value {
    display: block;
    color: #0F172A;
    font-size: 15px;
    font-weight: 800;
}

.tab-title {
    font-size: 18px;
    font-weight: 800;
    color: #0F172A;
    margin-bottom: 4px;
}

.kpi-card label {
    font-weight: 700 !important;
}
"""


with gr.Blocks(css=CUSTOM_CSS, title="AutoQuote AI Strategy Studio") as dashboard:

    gr.HTML(
        """
        <div id="main-header">
            <h1>AutoQuote AI — Personal Setup Studio</h1>
            <p>
                Describe what you want to do. The agent builds a compatible PC, compares strategy options,
                prepares a quotation, and highlights a few carefully selected value options.
            </p>
        </div>
        """
    )

    # Hidden tracking components
    with gr.Row(visible=False):
        original_total_kpi = gr.Number(visible=False, value=0)
        optimized_total_kpi = gr.Number(visible=False, value=0)

    # ============================================================
    # INPUT REGION
    # ============================================================

    with gr.Group(elem_classes=["input-card"]):
        with gr.Row():
            with gr.Column(scale=5):
                user_query = gr.Textbox(
                    label="What do you want this PC setup to do?",
                    placeholder="Example: I want a smooth AAA gaming PC for 1440p gameplay. Deliver to Petaling Jaya.",
                    lines=2
                )

            with gr.Column(scale=2):
                budget_input = gr.Number(
                    label="Comfort Budget (RM)",
                    value=4000,
                    precision=0
                )

            with gr.Column(scale=2):
                strategy_choice = gr.Dropdown(
                    choices=[
                        "🚀 Max Performance",
                        "⚖️ Optimal Value",
                        "📉 Budget Saver"
                    ],
                    value="⚖️ Optimal Value",
                    label="Strategy Focus"
                )

            with gr.Column(scale=2):
                gr.HTML("<br>")
                submit_btn = gr.Button(
                    "Build My Setup",
                    variant="primary",
                    size="lg"
                )

    gr.Markdown("")

    # ============================================================
    # MAIN RECOMMENDED BUILD STORY
    # ============================================================

    with gr.Group(elem_classes=["hero-section"]):
        gr.HTML(
            """
            <div class="hero-title">🎯 Your Recommended Build</div>
            <div class="hero-subtitle">
                A customer-friendly explanation of why this selected setup fits your needs, budget, and strategy focus.
            </div>
            """
        )

        output_explain = gr.Markdown(elem_classes=["scroll-box"])

    # ============================================================
    # CAREFULLY PICKED EXTRAS — VISIBLE OUTSIDE TABS
    # ============================================================

    with gr.Group(elem_classes=["extras-section"]):
        gr.HTML(
            """
            <div class="extras-title">💡 Carefully Picked Extras</div>
            <div class="hero-subtitle">
                A few high-signal suggestions that may make your setup more useful, complete, or future-ready.
            </div>
            """
        )

        recommendation_cards = gr.HTML()

        with gr.Accordion("View full recommendation details", open=False):
            recommendation_details_df = gr.Dataframe(
                show_label=False,
                interactive=False,
                wrap=True
            )

    # ============================================================
    # MAIN RESULT TABS
    # ============================================================

    with gr.Tabs():

        # --------------------------------------------------------
        # TAB 1: PRICE PROPOSAL
        # --------------------------------------------------------

        with gr.Tab("📄 Price Proposal"):
            gr.Markdown(
                """
                <div class="tab-title">Official Price Proposal</div>
                <div class="section-note">
                    This is the generated quotation for the selected strategy build.
                </div>
                """
            )

            output_quote = gr.Textbox(
                show_label=False,
                lines=16,
                max_lines=24,
                show_copy_button=True,
                elem_classes=["clean-invoice"]
            )

        # --------------------------------------------------------
        # TAB 2: STRATEGY COMPARISON
        # --------------------------------------------------------

        with gr.Tab("⚖️ Strategy Comparison"):
            gr.Markdown(
                """
                <div class="tab-title">Alternative Ways to Balance Your Budget</div>
                <div class="section-note">
                    Compare how each strategy balances total cost, estimated speed, and value efficiency.
                </div>
                """
            )

            strategy_summary_df = gr.Dataframe(
                show_label=False,
                interactive=False,
                wrap=True
            )

        # --------------------------------------------------------
        # TAB 3: TRUST & VERIFICATION
        # --------------------------------------------------------

        with gr.Tab("🛡️ Trust & Verification"):
            gr.Markdown(
                """
                <div class="tab-title">Trust & Verification</div>
                <div class="section-note">
                    This section shows what the agent understood, how much budget space remains,
                    and the compatibility evidence behind the selected build.
                </div>
                """
            )

            with gr.Row():
                savings_kpi = gr.Number(
                    label="Budget Space Remaining (RM)",
                    value=0,
                    precision=0,
                    elem_classes=["kpi-card"]
                )

                performance_kpi = gr.Number(
                    label="System Power Score",
                    value=0,
                    precision=1,
                    elem_classes=["kpi-card"]
                )

            with gr.Accordion("🔍 What We Understood", open=True):
                output_req = gr.HTML()

            with gr.Accordion("🛡️ Compatibility Check", open=False):
                output_compat = gr.Markdown()

            with gr.Accordion("📦 Full Parts Evidence", open=False):
                component_matrix_df = gr.Dataframe(
                    show_label=False,
                    interactive=False,
                    wrap=True
                )

    # ============================================================
    # DASHBOARD EXECUTION BINDING
    # ============================================================

    submit_btn.click(
        fn=process_sales_pipeline,
        inputs=[
            user_query,
            budget_input,
            strategy_choice
        ],
        outputs=[
            output_req,
            original_total_kpi,
            optimized_total_kpi,
            savings_kpi,
            performance_kpi,
            strategy_summary_df,
            component_matrix_df,
            recommendation_cards,
            recommendation_details_df,
            output_compat,
            output_quote,
            output_explain
        ]
    )


if __name__ == "__main__":
    dashboard.launch(debug=True, inline=True)

✅ CUSTOMER_NEXT_BUY_HISTORY loaded live from Google Sheets!
   Rows synced: 30
   Columns: ['case_id', 'trigger_category', 'trigger_item', 'use_case', 'budget_range', 'selected_build_type', 'next_product', 'next_category', 'next_price', 'accepted', 'reason']
✅ ADDON_CATALOG loaded live from Google Sheets!
   Rows synced: 26
   Columns: ['addon_id', 'addon_name', 'addon_category', 'suitable_use_case', 'recommended_for_build_type', 'trigger_gpu_tier', 'trigger_budget_range', 'price', 'priority', 'recommendation_section', 'stock', 'delivery_required', 'reason', 'benefit', 'upsell_message']


/tmp/ipykernel_1850/3723899940.py:847: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, title="AutoQuote AI Strategy Studio") as dashboard:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://15bbc067cebcb17b39.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



🤖 AUTONOMOUS SALES ENGINEER AGENT ACTIVE

📋 AGENT 1: REQUIREMENT AGENT

🔍 AGENT 2: SEARCH AGENT

🛡️ AGENT 3: COMPATIBILITY AGENT

⚡ AGENT 4: OPTIMIZER AGENT

💡 AGENT 5: NEXT-BEST-BUY & VALUE OPTIONS AGENT

✅ PIPELINE STATE GENERATED SUCCESSFULLY

🤖 AUTONOMOUS SALES ENGINEER AGENT ACTIVE

📋 AGENT 1: REQUIREMENT AGENT

🔍 AGENT 2: SEARCH AGENT

🛡️ AGENT 3: COMPATIBILITY AGENT

⚡ AGENT 4: OPTIMIZER AGENT

💡 AGENT 5: NEXT-BEST-BUY & VALUE OPTIONS AGENT

✅ PIPELINE STATE GENERATED SUCCESSFULLY

🤖 AUTONOMOUS SALES ENGINEER AGENT ACTIVE

📋 AGENT 1: REQUIREMENT AGENT

🔍 AGENT 2: SEARCH AGENT

🛡️ AGENT 3: COMPATIBILITY AGENT

⚡ AGENT 4: OPTIMIZER AGENT

💡 AGENT 5: NEXT-BEST-BUY & VALUE OPTIONS AGENT

✅ PIPELINE STATE GENERATED SUCCESSFULLY
